# 06 — Portfolio Feature Engineering

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

### Objective
Create a **point-in-time portfolio-level ML feature dataset** by combining the stock-level features from Notebook 03 with the portfolio weights/risk targets created in Notebooks 04–05.

### Core rule
**Every model feature must use information available on or before date `t`.**

Future fields (`Future_*` and target columns) are retained only for labels/audit and are never used as model features.


## 1. Inputs and outputs

### Inputs
- `data/processed/stock_features.csv`
- `data/processed/portfolio_stock_weights.csv`
- `data/processed/portfolio_daily_baseline.csv`
- `data/processed/portfolio_risk_targets.csv`

### Outputs
- `data/processed/portfolio_ml_features.csv`
- `reports/portfolio_feature_summary.csv`
- `reports/portfolio_feature_leakage_audit.csv`
- `reports/portfolio_feature_missingness.csv`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

DATA = Path("../data/processed")
REPORTS = Path("../reports")
REPORTS.mkdir(parents=True, exist_ok=True)

STOCK_FEATURES_PATH = DATA / "stock_features.csv"
WEIGHTS_PATH = DATA / "portfolio_stock_weights.csv"
PORTFOLIO_PATH = DATA / "portfolio_daily_baseline.csv"
TARGET_PATH = DATA / "portfolio_risk_targets.csv"


## 2. Load the existing project datasets


In [ ]:
stock = pd.read_csv(STOCK_FEATURES_PATH, parse_dates=["Date"])
weights = pd.read_csv(WEIGHTS_PATH, parse_dates=["Date"])
portfolio = pd.read_csv(PORTFOLIO_PATH, parse_dates=["Date"])
targets = pd.read_csv(TARGET_PATH, parse_dates=["Date"])

print("stock_features:", stock.shape)
print("portfolio_stock_weights:", weights.shape)
print("portfolio_daily_baseline:", portfolio.shape)
print("portfolio_risk_targets:", targets.shape)

print("\nDate ranges:")
for name, df in {
    "stock_features": stock,
    "weights": weights,
    "portfolio": portfolio,
    "targets": targets,
}.items():
    print(f"{name:28s}: {df['Date'].min()} -> {df['Date'].max()}")


## 3. Basic integrity checks

The portfolio weights were created from the daily-rebalanced equal-weight portfolio. We verify the expected identifiers and that the weight table is suitable for a point-in-time merge.


In [ ]:
required_stock_cols = {"Date", "Ticker", "Sector"}
required_weight_cols = {"Date", "Ticker", "Sector", "Weight"}
required_target_cols = {
    "Date", "Target_DD_3Pct_10D",
    "Target_DD_5Pct_10D", "Target_DD_10Pct_10D"
}

assert required_stock_cols.issubset(stock.columns), "Missing stock identifiers."
assert required_weight_cols.issubset(weights.columns), "Missing portfolio weight columns."
assert required_target_cols.issubset(targets.columns), "Missing target columns."

print("Duplicate stock Date-Ticker rows:",
      stock.duplicated(["Date", "Ticker"]).sum())
print("Duplicate weight Date-Ticker rows:",
      weights.duplicated(["Date", "Ticker"]).sum())
print("Duplicate portfolio dates:",
      portfolio["Date"].duplicated().sum())
print("Duplicate target dates:",
      targets["Date"].duplicated().sum())

weight_check = weights.groupby("Date")["Weight"].sum()
print("Maximum absolute weight-sum error:",
      (weight_check - 1).abs().max())


## 4. Restrict to the portfolio's eligible dates

Notebook 05 established the primary eligibility rule as **at least 40 stocks with valid same-day returns**. We use the already-created target dataset as the authoritative date universe.


In [ ]:
eligible_dates = targets["Date"].dropna().drop_duplicates().sort_values()

weights = weights[weights["Date"].isin(eligible_dates)].copy()
stock = stock[stock["Date"].isin(eligible_dates)].copy()
portfolio = portfolio[portfolio["Date"].isin(eligible_dates)].copy()
targets = targets[targets["Date"].isin(eligible_dates)].copy()

print("Eligible dates:", len(eligible_dates))
print("Date range:", eligible_dates.min(), "to", eligible_dates.max())


## 5. Merge point-in-time stock features with portfolio weights

Each stock feature is joined to its **same-day portfolio weight**. No future rows are used.


In [ ]:
feature_panel = weights[["Date", "Ticker", "Sector", "Weight"]].merge(
    stock,
    on=["Date", "Ticker", "Sector"],
    how="left",
    suffixes=("_portfolio", "_stock"),
    validate="one_to_one"
)

print("Merged feature panel:", feature_panel.shape)

if "Close" in feature_panel.columns:
    print(
        "Weight rows without matching stock features:",
        feature_panel["Close"].isna().sum()
    )


## 6. Identify stock-level feature columns

We exclude identifiers, raw price fields, and fields that are not appropriate for direct portfolio aggregation. The engineered stock signals from Notebook 03 are aggregated using current portfolio weights.


In [ ]:
exclude_from_stock_aggregation = {
    "Date", "Ticker", "Company_Name", "Sector",
    "Open", "High", "Low", "Close", "Volume",
    "Dividend", "Stock_Split",
    "Market_Return"
}

candidate_stock_features = [
    c for c in stock.columns
    if c not in exclude_from_stock_aggregation
    and pd.api.types.is_numeric_dtype(stock[c])
]

print("Number of stock features to aggregate:",
      len(candidate_stock_features))
print(candidate_stock_features)


## 7. Weighted portfolio-level stock signals

For each feature `x`:

`PortfolioFeature_t = sum_i(w_i,t * x_i,t)`

This converts stock-level momentum, volatility, trend, drawdown, volume and RSI signals into portfolio-level signals while respecting the portfolio's actual daily weights.


In [ ]:
weighted_features = feature_panel.copy()

for col in candidate_stock_features:
    weighted_features[col] = (
        weighted_features[col] * weighted_features["Weight"]
    )

portfolio_stock_features = (
    weighted_features.groupby("Date")[candidate_stock_features]
    .sum(min_count=1)
    .add_prefix("PortfolioWeighted_")
    .reset_index()
)

portfolio_stock_features.head()


## 8. Cross-sectional portfolio breadth and dispersion

Weighted averages alone can hide dispersion. We therefore create point-in-time breadth and dispersion measures.


In [ ]:
cross_sectional = feature_panel.groupby("Date").agg(
    Feature_Stock_Count=("Ticker", "nunique"),
    Positive_Return_1D_Ratio=("Return_1D", lambda x: (x > 0).mean()),
    Positive_Return_5D_Ratio=("Return_5D", lambda x: (x > 0).mean()),
    Positive_Return_20D_Ratio=("Return_20D", lambda x: (x > 0).mean()),
    Median_Return_1D=("Return_1D", "median"),
    Median_Return_5D=("Return_5D", "median"),
    Median_Return_20D=("Return_20D", "median"),
    Dispersion_Return_1D=("Return_1D", "std"),
    Dispersion_Return_5D=("Return_5D", "std"),
    Dispersion_Return_20D=("Return_20D", "std"),
    Dispersion_Volatility_20D=("Volatility_20D", "std"),
).reset_index()

cross_sectional.head()


## 9. Portfolio-level risk and concentration features

Notebook 04 already calculated portfolio-level risk and concentration measures. We reuse them rather than recomputing them.


In [ ]:
portfolio_feature_cols = [
    "Date", "Portfolio_Return", "Number_of_Stocks", "Weight_Check",
    "Portfolio_Value", "Running_Peak", "Current_Drawdown",
    "Volatility_5D", "Volatility_20D", "Volatility_60D",
    "Return_5D", "Return_20D", "Max_Drawdown_20D",
    "Max_Drawdown_60D", "Stock_HHI", "Largest_Stock_Weight",
    "Top_5_Stock_Weight", "Largest_Sector_Weight", "Sector_HHI"
]

portfolio_feature_cols = [
    c for c in portfolio_feature_cols if c in portfolio.columns
]

portfolio_risk_features = portfolio[portfolio_feature_cols].copy()
portfolio_risk_features.head()


## 10. Add current market regime information

`Market_Return` is already present in the stock-level feature dataset. Because it is the market return observed on date `t`, we aggregate it once per date and keep it as a current-information feature.


In [ ]:
if "Market_Return" in feature_panel.columns:
    market_features = (
        feature_panel.groupby("Date")
        .agg(Market_Return=("Market_Return", "first"))
        .reset_index()
    )
else:
    market_features = pd.DataFrame({
        "Date": eligible_dates,
        "Market_Return": np.nan
    })

market_features.head()


## 11. Combine all point-in-time features


In [ ]:
ml_features = portfolio_risk_features.merge(
    portfolio_stock_features,
    on="Date",
    how="left",
    validate="one_to_one"
).merge(
    cross_sectional,
    on="Date",
    how="left",
    validate="one_to_one"
).merge(
    market_features,
    on="Date",
    how="left",
    validate="one_to_one"
)

print("ML feature table shape before labels:", ml_features.shape)


## 12. Attach the risk targets

The target dataset contains both current features and future label-generation fields. We attach the target columns only. Future drawdown/value columns are kept separately for auditability and are explicitly excluded from model features.


In [ ]:
label_cols = [
    "Date",
    "Target_DD_3Pct_10D",
    "Target_DD_5Pct_10D",
    "Target_DD_10Pct_10D",
]

audit_label_cols = [
    "Date",
    "Future_Min_Value_10D",
    "Future_End_Value_10D",
    "Future_Drawdown_10D",
    "Future_Window_End",
]

labels = targets[
    [c for c in label_cols if c in targets.columns]
].copy()

audit_labels = targets[
    [c for c in audit_label_cols if c in targets.columns]
].copy()

ml_features = ml_features.merge(
    labels, on="Date", how="inner", validate="one_to_one"
)

audit_frame = ml_features[["Date"]].merge(
    audit_labels, on="Date", how="left", validate="one_to_one"
)

print("Final ML table shape:", ml_features.shape)
print("Target event counts:")

for c in [
    "Target_DD_3Pct_10D",
    "Target_DD_5Pct_10D",
    "Target_DD_10Pct_10D"
]:
    if c in ml_features.columns:
        print(f"{c}: {int(ml_features[c].sum())}")


## 13. Define the primary target

Notebook 05 selected **5% drawdown within the next 10 trading days** as the primary ML target.


In [ ]:
PRIMARY_TARGET = "Target_DD_5Pct_10D"

ml_features[PRIMARY_TARGET] = (
    ml_features[PRIMARY_TARGET].astype("Int64")
)

print("Primary target:", PRIMARY_TARGET)
print(
    ml_features[PRIMARY_TARGET]
    .value_counts(dropna=False)
    .sort_index()
)
print("Positive rate:", ml_features[PRIMARY_TARGET].mean())


## 14. Remove obvious leakage fields from the model feature set

The following are label-generation fields and must never be model inputs:

- `Future_Min_Value_10D`
- `Future_End_Value_10D`
- `Future_Drawdown_10D`
- `Future_Window_End`
- all target columns


In [ ]:
future_fields = {
    "Future_Min_Value_10D",
    "Future_End_Value_10D",
    "Future_Drawdown_10D",
    "Future_Window_End",
    "Target_DD_3Pct_10D",
    "Target_DD_5Pct_10D",
    "Target_DD_10Pct_10D",
}

identifier_fields = {"Date"}

feature_columns = [
    c for c in ml_features.columns
    if c not in future_fields | identifier_fields
    and pd.api.types.is_numeric_dtype(ml_features[c])
]

print("Number of candidate ML features:", len(feature_columns))
print(feature_columns)


## 15. Leakage audit

We check that no column name containing obvious future/target terminology is included in the candidate feature list.


In [ ]:
leakage_keywords = [
    "future", "target", "label", "forward", "next_", "lead_"
]

suspected_leakage = [
    c for c in feature_columns
    if any(k in c.lower() for k in leakage_keywords)
]

leakage_audit = pd.DataFrame({
    "Check": [
        "Future/target fields excluded",
        "Candidate feature count",
        "Suspected leakage feature count",
    ],
    "Result": [
        len(suspected_leakage) == 0,
        len(feature_columns),
        len(suspected_leakage),
    ]
})

print("Suspected leakage fields:", suspected_leakage)

assert len(suspected_leakage) == 0, (
    f"Potential leakage detected: {suspected_leakage}"
)

leakage_audit


## 16. Missingness analysis

Early observations can have missing long-window indicators such as 200-day moving averages. We quantify missingness before deciding how the modeling pipeline should handle it.


In [ ]:
missingness = (
    ml_features[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("Missing_Percent")
    .reset_index()
    .rename(columns={"index": "Feature"})
)

missingness.head(20)


## 17. Final modeling dataset

For the ML dataset we keep the date, all point-in-time features, and the three candidate targets. The **5% target** is the primary target for Notebook 07 onward.


In [ ]:
final_columns = (
    ["Date"]
    + feature_columns
    + [
        "Target_DD_3Pct_10D",
        "Target_DD_5Pct_10D",
        "Target_DD_10Pct_10D",
    ]
)

final_ml_dataset = ml_features[final_columns].copy()
final_ml_dataset = (
    final_ml_dataset
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Final dataset shape:", final_ml_dataset.shape)
print(
    "Date range:",
    final_ml_dataset["Date"].min(),
    "to",
    final_ml_dataset["Date"].max()
)
print("Unique dates:", final_ml_dataset["Date"].nunique())

final_ml_dataset.head()


## 18. Final quality checks


In [ ]:
assert final_ml_dataset["Date"].is_monotonic_increasing
assert final_ml_dataset["Date"].duplicated().sum() == 0
assert (
    final_ml_dataset[PRIMARY_TARGET]
    .dropna()
    .isin([0, 1])
    .all()
)
assert all(c not in feature_columns for c in future_fields)

print(
    "Duplicate dates:",
    final_ml_dataset["Date"].duplicated().sum()
)
print(
    "Missing primary target:",
    final_ml_dataset[PRIMARY_TARGET].isna().sum()
)
print("Feature count:", len(feature_columns))
print("Final quality checks: PASSED")


## 19. Feature summary report


In [ ]:
feature_summary = pd.DataFrame({
    "Feature": feature_columns,
    "Dtype": [
        str(final_ml_dataset[c].dtype)
        for c in feature_columns
    ],
    "Missing_Percent": [
        final_ml_dataset[c].isna().mean() * 100
        for c in feature_columns
    ],
    "Unique_Values": [
        final_ml_dataset[c].nunique(dropna=True)
        for c in feature_columns
    ],
    "Mean": [
        final_ml_dataset[c].mean()
        for c in feature_columns
    ],
    "Std": [
        final_ml_dataset[c].std()
        for c in feature_columns
    ],
    "Min": [
        final_ml_dataset[c].min()
        for c in feature_columns
    ],
    "Max": [
        final_ml_dataset[c].max()
        for c in feature_columns
    ],
})

feature_summary.head(20)


## 20. Save Notebook 06 outputs

The primary output is the portfolio-level ML feature dataset. The audit reports document missingness and leakage checks.


In [ ]:
OUTPUT_PATH = DATA / "portfolio_ml_features.csv"
SUMMARY_PATH = REPORTS / "portfolio_feature_summary.csv"
LEAKAGE_PATH = REPORTS / "portfolio_feature_leakage_audit.csv"
MISSINGNESS_PATH = REPORTS / "portfolio_feature_missingness.csv"

final_ml_dataset.to_csv(OUTPUT_PATH, index=False)
feature_summary.to_csv(SUMMARY_PATH, index=False)
leakage_audit.to_csv(LEAKAGE_PATH, index=False)
missingness.to_csv(MISSINGNESS_PATH, index=False)

print("Step 06 outputs saved:")
print(OUTPUT_PATH)
print(SUMMARY_PATH)
print(LEAKAGE_PATH)
print(MISSINGNESS_PATH)


# Final methodology

### Portfolio universe
Dates satisfying the Notebook 05 eligibility rule of at least **40 stocks with valid same-day returns**.

### Portfolio construction
Daily-rebalanced equal-weight portfolio from Notebook 04.

### Feature construction
Stock-level signals are aggregated using the portfolio's same-day weights. Cross-sectional breadth and dispersion are added alongside portfolio-level volatility, drawdown, return and concentration signals.

### Target
Primary target: **at least 5% portfolio drawdown within the next 10 trading days**.

### Leakage rule
**Features use information available by date `t`; targets describe what happens after date `t`.**

### Next notebook
`07_model_training_and_evaluation.ipynb` — time-based model development using chronological train/validation/test splits.
